In [ ]:
# In[1]:

import pandas as pd

# Load files
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (unit='s')
for df in (metric, trace, log, error_logs):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) For each file, report number of rows and min/max timestamp
def file_stats(df, name):
    rows = df.shape[0]
    if rows:
        return {'file': name, 'rows': rows, 'min_timestamp_utc': df['timestamp'].min(), 'max_timestamp_utc': df['timestamp'].max()}
    else:
        return {'file': name, 'rows': 0, 'min_timestamp_utc': pd.NaT, 'max_timestamp_utc': pd.NaT}

file_stats_df = pd.DataFrame([
    file_stats(metric, "metric.csv"),
    file_stats(trace, "trace.csv"),
    file_stats(log, "log.csv"),
    file_stats(error_logs, "error_logs.csv"),
])

# 2) For each file, list unique cmdb_id values (all of them)
unique_cmdbs = []
unique_cmdbs.append(pd.DataFrame({'file': 'metric.csv', 'cmdb_id': pd.Series(metric['cmdb_id'].dropna().unique())}))
unique_cmdbs.append(pd.DataFrame({'file': 'trace.csv',  'cmdb_id': pd.Series(trace['cmdb_id'].dropna().unique())}))
unique_cmdbs.append(pd.DataFrame({'file': 'log.csv',    'cmdb_id': pd.Series(log['cmdb_id'].dropna().unique())}))
unique_cmdbs.append(pd.DataFrame({'file': 'error_logs.csv','cmdb_id': pd.Series(error_logs['cmdb_id'].dropna().unique())}))
unique_cmdb_df = pd.concat(unique_cmdbs, ignore_index=True)

# 3) From metric.csv: list up to 50 distinct component-KPI pairs with their global P95 values
# ensure numeric values and drop NaNs
metric_vals = metric.dropna(subset=['value'])
metric_vals['value'] = pd.to_numeric(metric_vals['value'], errors='coerce')
metric_vals = metric_vals.dropna(subset=['value'])
metric_p95 = metric_vals.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index(name='p95_value')
metric_p95_top50_df = metric_p95.sort_values('p95_value', ascending=False).head(50).reset_index(drop=True)

# 4) From trace.csv: list up to 20 distinct trace_name values and the count of rows per trace_name
trace_counts_df = trace.groupby('trace_name').size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)

# 5) From log.csv: list up to 20 distinct log_name values and the count of rows per log_name
log_counts_df = log.groupby('log_name').size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)

# 6) From error_logs.csv: show up to 20 sample rows (timestamp as UTC datetime, cmdb_id, message) and the total count
error_sample_df = error_logs[['timestamp', 'cmdb_id', 'message']].head(20).reset_index(drop=True)
error_total_count_df = pd.DataFrame({'error_total_count': [error_logs.shape[0]]})

# Display compact results (variables separated by commas as IPython outputs)
file_stats_df, unique_cmdb_df, metric_p95_top50_df, trace_counts_df, log_counts_df, error_sample_df, error_total_count_df

```
Out[1]:
```
Summary of the telemetry inspection (UTC):

1) File sizes & time range
- metric.csv: 1,850 rows; timestamps from 2024-01-18 15:37:00+00:00 to 2024-01-18 16:01:00+00:00
- trace.csv: 3,720 rows; timestamps from 2024-01-18 15:37:00+00:00 to 2024-01-18 16:01:00+00:00
- log.csv: 452 rows; timestamps from 2024-01-18 15:37:00+00:00 to 2024-01-18 16:01:00+00:00
- error_logs.csv: 0 rows (no timestamps present)

2) Unique cmdb_id values (present across files)
Examples include: adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, productcatalogservice, recommendationservice, paymentservice, redis, shippingservice (and other Online Boutique services). These are the services observed in the telemetry sets.

3) Top component–KPI P95s from metric.csv (top 5 examples; full list limited to 50 is available in the previous output)
- adservice — mem — p95 ≈ 1.0862e+08 (highest)
- emailservice — mem — p95 ≈ 4.9893e+07
- currencyservice — mem — p95 ≈ 4.6555e+07
- recommendationservice — mem — p95 ≈ 4.5285e+07
- cartservice — mem — p95 ≈ 4.3512e+07
(Observation: memory-related KPIs dominate the highest P95 values. Lower P95 examples shown near the tail include CPU and latency KPIs with much smaller values.)

4) trace.csv summary (top trace_name counts; up to 20 shown)
Top trace_name groups and counts:
- trace.from_root.row_count: 171
- trace.from_root.error_rate: 171
- trace.from_root.duration_p95: 171
- trace.from_root.duration_mean: 171
- trace.from_frontendservice.* (row_count/error_rate/duration_*): 124 each
- trace.from_checkoutservice.*: 120 each
- trace.to_productcatalogservice.*: 98 each
- trace.to_recommendationservice.*: 75 each
(Trace data shows multiple per-service trace features; root, frontendservice, checkoutservice, productcatalogservice and recommendationservice are prominent.)

5) log.csv summary (up to 20 distinct log_name)
- log.error_count: 226 rows
- log.row_count: 226 rows

6) error_logs.csv
- No error log rows found (0 total). Sample rows: none.

Concise conclusions:
- All telemetry files (metric, trace, log) cover the same ~24-minute UTC window (2024-01-18 15:37–16:01).
- Memory KPIs (mem) show the highest P95 values; adservice memory P95 is the largest observed.
- Trace data is concentrated around root, frontendservice, checkoutservice, productcatalogservice, and recommendationservice traces.
- log.csv contains aggregated row_count and error_count metrics; error_logs.csv has no raw error messages.

The original code execution output of IPython Kernel is also provided below for reference:

(             file  rows         min_timestamp_utc         max_timestamp_utc
0      metric.csv  1850 2024-01-18 15:37:00+00:00 2024-01-18 16:01:00+00:00
1       trace.csv  3720 2024-01-18 15:37:00+00:00 2024-01-18 16:01:00+00:00
2         log.csv   452 2024-01-18 15:37:00+00:00 2024-01-18 16:01:00+00:00
3  error_logs.csv     0                       NaT                       NaT,           file          cmdb_id
0   metric.csv        adservice
1   metric.csv      cartservice
2   metric.csv  checkoutservice
3   metric.csv  currencyservice
4   metric.csv     emailservice
..         ...              ...
25     log.csv  shippingservice
26     log.csv  checkoutservice
27     log.csv     emailservice
28     log.csv   paymentservice
29     log.csv            redis

[30 rows x 2 columns],                   cmdb_id    kpi_name     p95_value
0               adservice         mem  1.086204e+08
1            emailservice         mem  4.989283e+07
2         currencyservice         mem  4.655511e+07
3   recommendationservice         mem  4.528472e+07
4             cartservice         mem  4.351215e+07
..                    ...         ...           ...
45        shippingservice         cpu  3.111788e-01
46                  redis         cpu  3.047734e-01
47        checkoutservice  latency-90  2.730930e-01
48               frontend  latency-90  1.831873e-01
49        checkoutservice  latency-50  1.515066e-01

[50 rows x 3 columns],                                       trace_name  count
0                      trace.from_root.row_count    171
1                     trace.from_root.error_rate    171
2                   trace.from_root.duration_p95    171
3                  trace.from_root.duration_mean    171
4           trace.from_frontendservice.row_count    124
5          trace.from_frontendservice.error_rate    124
6        trace.from_frontendservice.duration_p95    124
7       trace.from_frontendservice.duration_mean    124
8           trace.from_checkoutservice.row_count    120
9          trace.from_checkoutservice.error_rate    120
10       trace.from_checkoutservice.duration_p95    120
11      trace.from_checkoutservice.duration_mean    120
12  trace.to_productcatalogservice.duration_mean     98
13   trace.to_productcatalogservice.duration_p95     98
14     trace.to_productcatalogservice.error_rate     98
15      trace.to_productcatalogservice.row_count     98
16  trace.to_recommendationservice.duration_mean     75
17   trace.to_recommendationservice.duration_p95     75
18     trace.to_recommendationservice.error_rate     75
19      trace.to_recommendationservice.row_count     75,           log_name  count
0  log.error_count    226
1    log.row_count    226, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [],    error_total_count
0                  0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
# reuse existing 'metric' if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")
# 1) Parse timestamp as UTC datetimes (unit='s')
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# 2) Candidate cmdb_id values
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# restrict metric to candidates for threshold calculation
metric_cand = metric[metric['cmdb_id'].isin(candidates)].copy()
metric_cand['value'] = pd.to_numeric(metric_cand['value'], errors='coerce')

# 3) For each (cmdb_id, kpi_name) compute global P95 and P05 using full available series (no time filter)
quantiles = metric_cand.dropna(subset=['value']).groupby(['cmdb_id', 'kpi_name'])['value'].quantile([0.95, 0.05]).unstack(level=-1)
# unstack created columns 0.95 and 0.05; rename
quantiles = quantiles.rename(columns={0.95: 'p95', 0.05: 'p05'}).reset_index()

# total number of (cmdb_id,kpi_name) pairs examined
total_pairs = quantiles.shape[0]

# 4) Filter metric rows to the incident window (inclusive)
start = pd.to_datetime('2024-01-18 15:34:58+00:00', utc=True)
end   = pd.to_datetime('2024-01-18 16:04:58+00:00', utc=True)
metric_window = metric_cand[(metric_cand['timestamp'] >= start) & (metric_cand['timestamp'] <= end)].copy()

# merge windowed rows with global thresholds
mw = metric_window.merge(quantiles, on=['cmdb_id', 'kpi_name'], how='left')

# 4) For each (cmdb_id, kpi_name) within that window compute required anomaly stats
def agg_anoms(df):
    p95 = df['p95'].iloc[0]
    p05 = df['p05'].iloc[0]
    vals = df['value']
    ts = df['timestamp']
    high_mask = vals >= p95
    low_mask = vals <= p05
    high_count = int(high_mask.sum())
    low_count = int(low_mask.sum())
    high_first = ts[high_mask].min() if high_count > 0 else pd.NaT
    low_first = ts[low_mask].min() if low_count > 0 else pd.NaT
    high_max = vals.max() if len(vals) > 0 else pd.NA
    low_min = vals.min() if len(vals) > 0 else pd.NA
    return pd.Series({
        'p95': p95,
        'p05': p05,
        'high_anomaly_count': high_count,
        'high_first_ts_utc': high_first,
        'high_max_value': high_max,
        'low_anomaly_count': low_count,
        'low_first_ts_utc': low_first,
        'low_min_value': low_min
    })

if not mw.empty:
    grouped = mw.groupby(['cmdb_id', 'kpi_name']).apply(agg_anoms).reset_index()
else:
    # empty DataFrame with the expected columns
    grouped = pd.DataFrame(columns=['cmdb_id','kpi_name','p95','p05','high_anomaly_count','high_first_ts_utc','high_max_value','low_anomaly_count','low_first_ts_utc','low_min_value'])

# 5) Keep only rows where (high_anomaly_count + low_anomaly_count) > 0
grouped['total_anoms'] = grouped['high_anomaly_count'].astype(int) + grouped['low_anomaly_count'].astype(int)
result_df = grouped[grouped['total_anoms'] > 0].copy()

# sort by total_anoms desc then cmdb_id; limit to top 20
result_df = result_df.sort_values(['total_anoms', 'cmdb_id'], ascending=[False, True]).head(20)

# select and order columns as requested
result_df = result_df[['cmdb_id','kpi_name','p95','p05',
                       'high_anomaly_count','high_first_ts_utc','high_max_value',
                       'low_anomaly_count','low_first_ts_utc','low_min_value']]

# Display compact results: result_df (top rows) and total_pairs count
result_df, pd.DataFrame({'total_pairs_examined': [total_pairs]})

```
Out[2]:
```
Summary (metric.csv, candidates checked: checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice)

- Total distinct (cmdb_id, kpi_name) pairs examined: 36.

- Major finding: many diskio and socket KPIs are constant and equal to their global P95/P05, producing large counts of both high and low "anomalies":
  - For these services: checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice
    - KPI diskio: p95 = p05 = 0.0, high_anomaly_count = 25, low_anomaly_count = 25, first anomaly ts = 2024-01-18 15:37:00+00:00, values = 0.0
    - KPI socket: p95 = p05 = integer (service-specific: checkout=9, currency=4, email=3, productcatalog=4, recommendation=4), high_anomaly_count = 25, low_anomaly_count = 25, first anomaly ts = 2024-01-18 15:37:00+00:00, values equal to that socket value
  - productcatalogservice also shows error KPI with p95=p05=0.0 and 25/25 anomalies (first ts 2024-01-18 15:37:00+00:00).

- Other KPIs with non-trivial anomalies (smaller counts, typically 2 high and 2 low per pair):
  - checkoutservice:
    - cpu: p95 ≈ 0.4968, p05 ≈ 0.33995; high_anomaly_count=2 (first high 2024-01-18 15:41:00, high_max ≈ 0.5565); low_anomaly_count=2 (first low 2024-01-18 15:40:00, low_min ≈ 0.3331)
    - latency-50: p95 ≈ 0.1515, p05 ≈ 0.0809; high_count=2 (first high 2024-01-18 16:00:00), low_count=2 (first low 2024-01-18 15:44:00)
    - latency-90: p95 ≈ 0.2731, p05 ≈ 0.2167; high_count=2 (first high 2024-01-18 15:53:00), low_count=2 (first low 2024-01-18 15:37:00)
    - mem: p95 ≈ 11,889,980, p05 ≈ 11,517,710; high_count=2 (first high 2024-01-18 15:51:00, high_max ≈ 12,055,360), low_count=2 (first low 2024-01-18 15:56:00, low_min ≈ 11,441,150)
    - workload: p95 ≈ 1.194, p05 ≈ 0.7869; high_count=2 (first high 2024-01-18 15:37:00), low_count=2 (first low 2024-01-18 15:40:00)
  - currencyservice:
    - cpu: p95 ≈ 31.727, p05 ≈ 28.1606; high_count=2 (first high 2024-01-18 15:51:00, high_max ≈ 32.338), low_count=2 (first low 2024-01-18 15:58:00, low_min ≈ 27.106)
    - latency-50: p95 ≈ 0.0036296, p05 ≈ 0.0034778; high_count=2 (first high 2024-01-18 15:37:00), low_count=2 (first low 2024-01-18 15:57:00)
    - latency-90: p95 ≈ 0.0365685, p05 ≈ 0.0215329; high_count=2 (first high 2024-01-18 15:43:00), low_count=2 (first low 2024-01-18 15:57:00)
    - mem: p95 ≈ 46,555,110, p05 ≈ 45,823,320; high_count=2 (first high 2024-01-18 15:39:00), low_count=2 (first low 2024-01-18 16:00:00)
  - (Similar small-count anomaly patterns appear for other candidate service KPIs listed in the result set.)

- Sorting by anomaly volume shows diskio/socket (and productcatalog error) dominate (50 anomalies each: 25 high + 25 low), followed by the CPU/memory/latency/workload KPIs with small numbers of anomalies (mostly 2 highs and 2 lows).

Interpretation:
- Disk I/O and socket metrics are uniformly equal to their global quantiles across the window for all candidate services (causing every point to be flagged as both high and low relative to p95/p05). This suggests those KPIs are constant-valued in the window (likely instrument normalization), not necessarily meaningful bursts.
- The more meaningful deviations (cpu, mem, latency, workload) show sparse anomalies (usually 2 points above P95 and 2 below P05) with first anomalous timestamps between 2024-01-18 15:37 and 16:00 UTC — these are the signals to inspect further per service.

If you want, I can return the compact result table again or focus on a specific service/KPI for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name           p95           p05  high_anomaly_count         high_first_ts_utc  high_max_value  low_anomaly_count          low_first_ts_utc  low_min_value
1         checkoutservice      diskio  0.000000e+00  0.000000e+00                  25 2024-01-18 15:37:00+00:00    0.000000e+00                 25 2024-01-18 15:37:00+00:00   0.000000e+00
5         checkoutservice      socket  9.000000e+00  9.000000e+00                  25 2024-01-18 15:37:00+00:00    9.000000e+00                 25 2024-01-18 15:37:00+00:00   9.000000e+00
8         currencyservice      diskio  0.000000e+00  0.000000e+00                  25 2024-01-18 15:37:00+00:00    0.000000e+00                 25 2024-01-18 15:37:00+00:00   0.000000e+00
12        currencyservice      socket  4.000000e+00  4.000000e+00                  25 2024-01-18 15:37:00+00:00    4.000000e+00                 25 2024-01-18 15:37:00+00:00   4.000000e+00
15           emailservice      diskio  0.000000e+00  0.000000e+00                  25 2024-01-18 15:37:00+00:00    0.000000e+00                 25 2024-01-18 15:37:00+00:00   0.000000e+00
19           emailservice      socket  3.000000e+00  3.000000e+00                  25 2024-01-18 15:37:00+00:00    3.000000e+00                 25 2024-01-18 15:37:00+00:00   3.000000e+00
22  productcatalogservice      diskio  0.000000e+00  0.000000e+00                  25 2024-01-18 15:37:00+00:00    0.000000e+00                 25 2024-01-18 15:37:00+00:00   0.000000e+00
23  productcatalogservice       error  0.000000e+00  0.000000e+00                  25 2024-01-18 15:37:00+00:00    0.000000e+00                 25 2024-01-18 15:37:00+00:00   0.000000e+00
27  productcatalogservice      socket  4.000000e+00  4.000000e+00                  25 2024-01-18 15:37:00+00:00    4.000000e+00                 25 2024-01-18 15:37:00+00:00   4.000000e+00
30  recommendationservice      diskio  0.000000e+00  0.000000e+00                  25 2024-01-18 15:37:00+00:00    0.000000e+00                 25 2024-01-18 15:37:00+00:00   0.000000e+00
34  recommendationservice      socket  4.000000e+00  4.000000e+00                  25 2024-01-18 15:37:00+00:00    4.000000e+00                 25 2024-01-18 15:37:00+00:00   4.000000e+00
0         checkoutservice         cpu  4.967575e-01  3.399521e-01                   2 2024-01-18 15:41:00+00:00    5.565025e-01                  2 2024-01-18 15:40:00+00:00   3.330723e-01
2         checkoutservice  latency-50  1.515066e-01  8.090662e-02                   2 2024-01-18 16:00:00+00:00    1.603774e-01                  2 2024-01-18 15:44:00+00:00   7.887294e-02
3         checkoutservice  latency-90  2.730930e-01  2.166835e-01                   2 2024-01-18 15:53:00+00:00    3.075800e-01                  2 2024-01-18 15:37:00+00:00   2.155000e-01
4         checkoutservice         mem  1.188998e+07  1.151771e+07                   2 2024-01-18 15:51:00+00:00    1.205536e+07                  2 2024-01-18 15:56:00+00:00   1.144115e+07
6         checkoutservice    workload  1.194000e+00  7.868533e-01                   2 2024-01-18 15:37:00+00:00    1.275550e+00                  2 2024-01-18 15:40:00+00:00   6.599833e-01
7         currencyservice         cpu  3.172737e+01  2.816062e+01                   2 2024-01-18 15:51:00+00:00    3.233812e+01                  2 2024-01-18 15:58:00+00:00   2.710632e+01
9         currencyservice  latency-50  3.629598e-03  3.477842e-03                   2 2024-01-18 15:37:00+00:00    3.818398e-03                  2 2024-01-18 15:57:00+00:00   3.456962e-03
10        currencyservice  latency-90  3.656852e-02  2.153287e-02                   2 2024-01-18 15:43:00+00:00    3.717994e-02                  2 2024-01-18 15:57:00+00:00   2.026233e-02
11        currencyservice         mem  4.655511e+07  4.582332e+07                   2 2024-01-18 15:39:00+00:00    4.669406e+07                  2 2024-01-18 16:00:00+00:00   4.579471e+07,    total_pairs_examined
0                    36)```
```

In [ ]:
# In[3]:

import re
import pandas as pd

# reuse existing 'trace' if available; otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# 1) Parse timestamp as UTC datetimes (unit='s')
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# ensure numeric values
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# 2) Compute global P95 and P05 for each distinct trace_name using full series
trace_q = trace.dropna(subset=['value']).groupby('trace_name')['value'].quantile([0.95, 0.05]).unstack().rename(columns={0.95: 'p95', 0.05: 'p05'}).reset_index()

# total distinct trace_name values examined
total_trace_names = trace['trace_name'].nunique()

# 3) Filter to incident window (inclusive)
start = pd.to_datetime('2024-01-18 15:34:58+00:00', utc=True)
end   = pd.to_datetime('2024-01-18 16:04:58+00:00', utc=True)
trace_window = trace[(trace['timestamp'] >= start) & (trace['timestamp'] <= end)].copy()

# 4) Merge windowed rows with global thresholds
tw = trace_window.merge(trace_q, on='trace_name', how='left')

# if no rows in window, prepare empty result
if tw.empty:
    result_df = pd.DataFrame(columns=['trace_name','direction','service','p95','p05','high_anomaly_count','high_first_ts_utc','low_anomaly_count','low_first_ts_utc'])
    matching_count = 0
else:
    # 5) extract direction and service from trace_name using patterns 'from_<service>.' or 'to_<service>.'
    # pattern: find 'from_' or 'to_' after 'trace.' and capture until next dot
    m = tw['trace_name'].str.extract(r'trace\.(from|to)_([^.]+)\.', expand=True)
    tw['direction'] = m[0]
    tw['service'] = m[1]

    # candidate services filter
    candidates = {'checkoutservice','currencyservice','emailservice','productcatalogservice','recommendationservice'}

    # 4) For each trace_name compute anomaly stats within window
    def agg_trace(df):
        p95 = df['p95'].iloc[0]
        p05 = df['p05'].iloc[0]
        vals = df['value']
        ts = df['timestamp']
        high_mask = vals >= p95
        low_mask = vals <= p05
        high_count = int(high_mask.sum())
        low_count = int(low_mask.sum())
        high_first = ts[high_mask].min() if high_count>0 else pd.NaT
        low_first = ts[low_mask].min() if low_count>0 else pd.NaT
        return pd.Series({
            'p95': p95,
            'p05': p05,
            'high_anomaly_count': high_count,
            'high_first_ts_utc': high_first,
            'low_anomaly_count': low_count,
            'low_first_ts_utc': low_first
        })

    agg = tw.groupby('trace_name').apply(agg_trace).reset_index()

    # bring direction/service (take first non-null per trace_name)
    svc = tw.groupby('trace_name').agg({'direction': lambda x: x.dropna().iloc[0] if x.dropna().size>0 else pd.NA,
                                        'service':   lambda x: x.dropna().iloc[0] if x.dropna().size>0 else pd.NA}).reset_index()

    merged = agg.merge(svc, on='trace_name', how='left')[['trace_name','direction','service','p95','p05','high_anomaly_count','high_first_ts_utc','low_anomaly_count','low_first_ts_utc']]

    # 6) Restrict to rows referencing candidate services in extracted service field
    filtered = merged[merged['service'].isin(candidates)].copy()

    # count how many trace_name rows match the candidate service filter
    matching_count = filtered['trace_name'].nunique()

    # 7) Keep only rows where (high_anomaly_count + low_anomaly_count) > 0
    filtered['total_anoms'] = filtered['high_anomaly_count'].astype(int) + filtered['low_anomaly_count'].astype(int)
    result_df = filtered[filtered['total_anoms'] > 0].copy()

    # sort and limit to top 20
    result_df = result_df.sort_values(['total_anoms','trace_name'], ascending=[False, True]).head(20)

    # final column order
    result_df = result_df[['trace_name','direction','service','p95','p05','high_anomaly_count','high_first_ts_utc','low_anomaly_count','low_first_ts_utc']]

# compact summary of totals
totals_df = pd.DataFrame({
    'total_distinct_trace_name_examined': [total_trace_names],
    'trace_name_matching_candidate_services': [matching_count]
})

# Return compact results
result_df, totals_df

```
Out[3]:
```
Summary (trace.csv, UTC):

- Totals:
  - Distinct trace_name examined: 44
  - trace_name rows referencing candidate services: 28

- Highest-volume anomalies (many points flagged because p95==p05==0 for error rates):
  - trace.from_checkoutservice.error_rate — p95=0, p05=0 — high=120, low=120 — first anomaly 2024-01-18 15:38:00+00:00
  - trace.to_productcatalogservice.error_rate — p95=0, p05=0 — high=98, low=98 — first 15:37:00+00:00
  - trace.to_recommendationservice.error_rate — p95=0, p05=0 — high=75, low=75 — first 15:37:00+00:00
  - trace.to_currencyservice.error_rate — p95=0, p05=0 — high=73, low=73 — first 15:37:00+00:00
  - trace.to_checkoutservice.error_rate — p95=0, p05=0 — high=72, low=72 — first 15:38:00+00:00
  - trace.from_recommendationservice.error_rate — p95=0, p05=0 — high=50, low=50 — first 15:37:00+00:00
  - trace.to_emailservice.error_rate — p95=0, p05=0 — high=49, low=49 — first 15:37:00+00:00

- Row-count and latency anomalies (smaller counts but more informative):
  - trace.to_productcatalogservice.row_count — p95≈3749.6, p05=24 — high=5 (first high 15:40), low=25 (first low 15:37)
  - trace.to_currencyservice.row_count — p95≈2353.6, p05=12 — high=4 (first high 15:52), low=24 (first low 15:38)
  - trace.from_checkoutservice.row_count — p95=260, p05=25 — high=7 (first high 15:40), low=10 (first low 15:38)
  - trace.from_checkoutservice.duration_mean / duration_p95 and trace.to_checkoutservice.duration_mean / duration_p95 show moderate anomaly counts (4–6 highs and lows) with first anomaly timestamps in the 15:38–15:50 range.
  - trace.to_currencyservice.duration_* also shows 4 highs and 4 lows with first anomalies ~15:40–15:45.

- Interpretation:
  - Error-rate traces for many services are constant-zero in global quantiles, which makes every window point equal to p95/p05 and get flagged en masse — this likely indicates a zero-valued/error-rate baseline rather than a real spike.
  - More actionable signals are the row_count and duration traces: several services (checkoutservice, productcatalogservice, currencyservice, recommendationservice) show both high and low deviations within the incident window, with first anomalous timestamps clustered around 15:37–15:52 UTC.

- Next steps (recommended):
  - Investigate trace.row_count and duration anomalies for checkoutservice, productcatalogservice, currencyservice, and recommendationservice (timestamps: start inspecting ~15:37–15:52 UTC).
  - Treat error_rate mass flags cautiously (they appear driven by zero-baseline quantiles); confirm whether zeros represent no errors or missing data.

The original code execution output of IPython Kernel is also provided below for reference:

(                                      trace_name direction                service          p95         p05  high_anomaly_count         high_first_ts_utc  low_anomaly_count          low_first_ts_utc
2          trace.from_checkoutservice.error_rate      from        checkoutservice     0.000000    0.000000                 120 2024-01-18 15:38:00+00:00                120 2024-01-18 15:38:00+00:00
38     trace.to_productcatalogservice.error_rate        to  productcatalogservice     0.000000    0.000000                  98 2024-01-18 15:37:00+00:00                 98 2024-01-18 15:37:00+00:00
42     trace.to_recommendationservice.error_rate        to  recommendationservice     0.000000    0.000000                  75 2024-01-18 15:37:00+00:00                 75 2024-01-18 15:37:00+00:00
22           trace.to_currencyservice.error_rate        to        currencyservice     0.000000    0.000000                  73 2024-01-18 15:37:00+00:00                 73 2024-01-18 15:37:00+00:00
18           trace.to_checkoutservice.error_rate        to        checkoutservice     0.000000    0.000000                  72 2024-01-18 15:38:00+00:00                 72 2024-01-18 15:38:00+00:00
10   trace.from_recommendationservice.error_rate      from  recommendationservice     0.000000    0.000000                  50 2024-01-18 15:37:00+00:00                 50 2024-01-18 15:37:00+00:00
26              trace.to_emailservice.error_rate        to           emailservice     0.000000    0.000000                  49 2024-01-18 15:37:00+00:00                 49 2024-01-18 15:37:00+00:00
39      trace.to_productcatalogservice.row_count        to  productcatalogservice  3749.600000   24.000000                   5 2024-01-18 15:40:00+00:00                 25 2024-01-18 15:37:00+00:00
23            trace.to_currencyservice.row_count        to        currencyservice  2353.600000   12.000000                   4 2024-01-18 15:52:00+00:00                 24 2024-01-18 15:38:00+00:00
3           trace.from_checkoutservice.row_count      from        checkoutservice   260.000000   25.000000                   7 2024-01-18 15:40:00+00:00                 10 2024-01-18 15:38:00+00:00
0       trace.from_checkoutservice.duration_mean      from        checkoutservice     0.009192    0.000017                   6 2024-01-18 15:50:00+00:00                  6 2024-01-18 15:43:00+00:00
1        trace.from_checkoutservice.duration_p95      from        checkoutservice     0.071048    0.000025                   6 2024-01-18 15:50:00+00:00                  6 2024-01-18 15:44:00+00:00
36  trace.to_productcatalogservice.duration_mean        to  productcatalogservice     0.004899    0.000016                   5 2024-01-18 15:38:00+00:00                  5 2024-01-18 15:37:00+00:00
37   trace.to_productcatalogservice.duration_p95        to  productcatalogservice     0.013926    0.000024                   5 2024-01-18 15:40:00+00:00                  5 2024-01-18 15:37:00+00:00
19            trace.to_checkoutservice.row_count        to        checkoutservice   271.450000   25.000000                   4 2024-01-18 15:42:00+00:00                  5 2024-01-18 15:38:00+00:00
11    trace.from_recommendationservice.row_count      from  recommendationservice   591.000000  520.000000                   4 2024-01-18 15:40:00+00:00                  4 2024-01-18 15:37:00+00:00
16        trace.to_checkoutservice.duration_mean        to        checkoutservice     0.132270    0.003950                   4 2024-01-18 15:51:00+00:00                  4 2024-01-18 15:40:00+00:00
17         trace.to_checkoutservice.duration_p95        to        checkoutservice     0.226361    0.006357                   4 2024-01-18 15:41:00+00:00                  4 2024-01-18 15:42:00+00:00
20        trace.to_currencyservice.duration_mean        to        currencyservice     0.001466    0.000050                   4 2024-01-18 15:43:00+00:00                  4 2024-01-18 15:45:00+00:00
21         trace.to_currencyservice.duration_p95        to        currencyservice     0.000264    0.000065                   4 2024-01-18 15:40:00+00:00                  4 2024-01-18 15:38:00+00:00,    total_distinct_trace_name_examined  trace_name_matching_candidate_services
0                                  44                                      28)```
```

In [ ]:
# In[4]:

import pandas as pd

# reuse existing variables if present, otherwise load
try:
    log
except NameError:
    log = pd.read_csv("log.csv")
try:
    error_logs
except NameError:
    error_logs = pd.read_csv("error_logs.csv")

# 1) Parse timestamp as UTC datetimes (unit='s')
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True) if 'timestamp' in error_logs.columns else pd.Series(dtype='datetime64[ns, UTC]')

# 2) Candidate cmdb_id values
candidates = ['checkoutservice','currencyservice','emailservice','productcatalogservice','recommendationservice']

# restrict log to candidate cmdb_ids for threshold calculation
log_cand = log[log['cmdb_id'].isin(candidates)].copy()
log_cand['value'] = pd.to_numeric(log_cand['value'], errors='coerce')

# 3) Compute global P95 and P05 for each (cmdb_id, log_name) using full series
quant = log_cand.dropna(subset=['value']).groupby(['cmdb_id','log_name'])['value'].quantile([0.95, 0.05]).unstack().rename(columns={0.95:'p95', 0.05:'p05'}).reset_index()

# total pairs examined
total_pairs = quant.shape[0]

# 4) Filter log rows to incident window (inclusive)
start = pd.to_datetime('2024-01-18 15:34:58+00:00', utc=True)
end   = pd.to_datetime('2024-01-18 16:04:58+00:00', utc=True)
log_window = log_cand[(log_cand['timestamp'] >= start) & (log_cand['timestamp'] <= end)].copy()

# 5) Merge window rows with global thresholds
lw = log_window.merge(quant, on=['cmdb_id','log_name'], how='left')

# aggregate anomaly stats per (cmdb_id, log_name)
if lw.empty:
    result_df = pd.DataFrame(columns=['cmdb_id','log_name','p95','p05',
                                      'high_anomaly_count','high_first_ts_utc','high_max_value',
                                      'low_anomaly_count','low_first_ts_utc','low_min_value'])
else:
    def agg_logs(df):
        p95 = df['p95'].iloc[0]
        p05 = df['p05'].iloc[0]
        vals = df['value']
        ts = df['timestamp']
        high_mask = vals >= p95
        low_mask = vals <= p05
        high_count = int(high_mask.sum())
        low_count = int(low_mask.sum())
        high_first = ts[high_mask].min() if high_count>0 else pd.NaT
        low_first = ts[low_mask].min() if low_count>0 else pd.NaT
        high_max = vals.max() if len(vals)>0 else pd.NA
        low_min = vals.min() if len(vals)>0 else pd.NA
        return pd.Series({
            'p95': p95,
            'p05': p05,
            'high_anomaly_count': high_count,
            'high_first_ts_utc': high_first,
            'high_max_value': high_max,
            'low_anomaly_count': low_count,
            'low_first_ts_utc': low_first,
            'low_min_value': low_min
        })

    agg = lw.groupby(['cmdb_id','log_name']).apply(agg_logs).reset_index()

    # 6) Keep only rows where (high + low) > 0
    agg['total_anoms'] = agg['high_anomaly_count'].astype(int) + agg['low_anomaly_count'].astype(int)
    result_df = agg[agg['total_anoms'] > 0].copy()

    # 7) Sort and limit to top 20 rows
    result_df = result_df.sort_values(['total_anoms','cmdb_id'], ascending=[False, True]).head(20)

    # select requested columns and order
    result_df = result_df[['cmdb_id','log_name','p95','p05',
                           'high_anomaly_count','high_first_ts_utc','high_max_value',
                           'low_anomaly_count','low_first_ts_utc','low_min_value']]

# 8) error_logs count (raw messages)
error_logs_count = error_logs.shape[0] if error_logs is not None else 0

totals_df = pd.DataFrame({
    'total_pairs_examined': [total_pairs],
    'error_logs_count': [error_logs_count]
})

# Return compact results
result_df, totals_df

```
Out[4]:
```
Summary (log.csv, UTC):

- Totals:
  - (cmdb_id, log_name) pairs examined for candidates: 8
  - Raw error_logs messages: 0

- Key findings:
  - log.error_count for multiple services is constant-zero relative to global quantiles, causing every point in the window to equal p95=p05 and thus be counted as both high and low:
    - checkoutservice — log.error_count — p95=0, p05=0 — high=24 / low=24 — first flagged 2024-01-18 15:38:00+00:00
    - currencyservice — log.error_count — high=25 / low=25 — first 15:37:00
    - emailservice — log.error_count — high=24 / low=24 — first 15:38:00
    - recommendationservice — log.error_count — high=25 / low=25 — first 15:37:00
  - These mass flags likely reflect zero-baseline values (instrument baseline) rather than actionable error spikes.

- More actionable anomalies: log.row_count (small number of high/low deviations)
  - checkoutservice — log.row_count — p95≈114.45 / p05≈69.9 — high=2 (first high 15:42, max=120) / low=2 (first low 15:47, min=63)
  - currencyservice — log.row_count — p95≈2524.40 / p05≈2216.4 — high=2 (first high 15:52, max=2552) / low=2 (first low 15:37, min=70)
  - emailservice — log.row_count — p95≈38.10 / p05≈23.3 — high=2 (first high 15:42, max=40) / low=2 (first low 15:47, min=21)
  - recommendationservice — log.row_count — p95≈590.80 / p05≈524.6 — high=2 (first high 15:40, max=597) / low=2 (first low 15:37, min=15)

- Interpretation / next steps (concise):
  - Treat log.error_count mass flags cautiously (zeros likely baseline).  
  - Investigate log.row_count deviations for checkoutservice, currencyservice, emailservice, recommendationservice (first anomalous timestamps between ~15:37 and 15:52 UTC) — these indicate throughput/traffic changes worth deeper inspection.  
  - No raw error messages available in error_logs.csv to correlate.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id         log_name      p95     p05  high_anomaly_count         high_first_ts_utc  high_max_value  low_anomaly_count          low_first_ts_utc  low_min_value
2        currencyservice  log.error_count     0.00     0.0                  25 2024-01-18 15:37:00+00:00               0                 25 2024-01-18 15:37:00+00:00              0
6  recommendationservice  log.error_count     0.00     0.0                  25 2024-01-18 15:37:00+00:00               0                 25 2024-01-18 15:37:00+00:00              0
0        checkoutservice  log.error_count     0.00     0.0                  24 2024-01-18 15:38:00+00:00               0                 24 2024-01-18 15:38:00+00:00              0
4           emailservice  log.error_count     0.00     0.0                  24 2024-01-18 15:38:00+00:00               0                 24 2024-01-18 15:38:00+00:00              0
1        checkoutservice    log.row_count   114.45    69.9                   2 2024-01-18 15:42:00+00:00             120                  2 2024-01-18 15:47:00+00:00             63
3        currencyservice    log.row_count  2524.40  2216.4                   2 2024-01-18 15:52:00+00:00            2552                  2 2024-01-18 15:37:00+00:00             70
5           emailservice    log.row_count    38.10    23.3                   2 2024-01-18 15:42:00+00:00              40                  2 2024-01-18 15:47:00+00:00             21
7  recommendationservice    log.row_count   590.80   524.6                   2 2024-01-18 15:40:00+00:00             597                  2 2024-01-18 15:37:00+00:00             15,    total_pairs_examined  error_logs_count
0                     8                 0)```
```